In [1]:
import ast
import os
import mne
from pathlib import Path
from utils.config import BASE_DIR, EAR_ELECTRODES, FACE_ELECTRODES, NECK_ELECTRODES
from utils.utils import compare_raw_and_file_annotations, merge_consecutive_annotations
from step0_mff_cleaning import add_sleep_scoring

mne.set_log_level("error")
# Set MNE configuration for light theme
mne.set_config('MNE_BROWSER_THEME', 'light')  # Set browser theme to light
mne.viz.set_browser_backend('qt')
# Alternative approach: set Qt style
os.environ['QT_STYLE_OVERRIDE'] = 'Fusion'  # Use Fusion style which is lighter

In [2]:
# Get all subjects with directories under MCI_clean
MCI_clean_path = Path(f"{BASE_DIR}/MCI_clean/a_the_rest")
subjects_in_MCI_clean = set([d.name for d in MCI_clean_path.iterdir() if d.is_dir()])

# Get all subjects with directories under new_MCI_results
new_MCI_results_path = Path("../results/new_MCI_results")
subjects_in_new_MCI = set([d.name for d in new_MCI_results_path.iterdir() if d.is_dir()]) if new_MCI_results_path.exists() else set()

# Find subjects in MCI_clean but not in new_MCI_results
missing_subjects = sorted(subjects_in_MCI_clean - subjects_in_new_MCI)

print(f"\nSubjects missing from new_MCI_results ({len(missing_subjects)}):")
for sub in missing_subjects:
    print(sub)


Subjects missing from new_MCI_results (7):
KS5
SC5
SM0016
SM0018
SM0019
SM0020
SM004


In [4]:
sub = "SM09"
sub_dir = f"{BASE_DIR}/MCI_clean/a_the_rest/{sub}"
filename = f"{sub}_cleaned_no_avg_ref_raw"
old_bad_channels = f"{sub_dir}/{sub}_bad_channels.txt"
new_bad_channels = f"{sub_dir}/{sub}_bad_channels.txt"
loaded_bad_channels = []
hypno_dir = "I:/Shaked/ISO_data/scoring/MCI"

In [5]:
raw = mne.io.read_raw(f"{sub_dir}/{filename}.fif", preload=False)
print(len(raw.ch_names))
# channels = [ch for ch in raw.ch_names if ch not in EAR_ELECTRODES]
# raw.pick(channels)        

176


In [6]:
# add_sleep_scoring(raw, sub, epoch_duration=30)
print([str(x) for x in set(raw.annotations.description)])
print(raw.info['bads'])
print("duration:", raw.times[-1]/60/60, "hours")

['NREM1', 'BAD', 'BAD_ACQ_SKIP', 'NREM3', 'REM', 'NREM2', 'UNKNOWN', 'Wake']
[]
duration: 6.28896111111111 hours


In [7]:
def sort_electrode_names(electrode_list):
    return sorted(electrode_list, key=lambda x: int(x[1:]) if x.startswith('E') and x[1:].isdigit() else float('inf'))

In [8]:
print("annotations.txt")
print("-" * 20)
compare_raw_and_file_annotations(raw, f"{sub_dir}/{sub}_annotations.txt")
# print("\ncleaned_annotations.txt")
# print("-" * 20)
# _ = compare_raw_and_file_annotations(raw, f"F:/gennadiy/ella_processed/{sub}/CleaningPipe/{sub}_cleaned_annotations.txt")

annotations.txt
--------------------
Annotations file not found: I:/Shaked/ISO_data/MCI_clean/a_the_rest/SM09/SM09_annotations.txt


In [9]:
# Load cleaned annotations if they exist
cleaned_annotations_path = f"{sub_dir}/{sub}_cleaned_annotations.txt"
if os.path.exists(cleaned_annotations_path):
    cleaned_annotations = mne.read_annotations(cleaned_annotations_path)
    raw.set_annotations(cleaned_annotations)
    print(f"Loaded {len(cleaned_annotations)} annotations from: {cleaned_annotations_path}")
else:
    print(f"Cleaned annotations file not found: {cleaned_annotations_path}")

Loaded 201 annotations from: I:/Shaked/ISO_data/MCI_clean/a_the_rest/SM09/SM09_cleaned_annotations.txt


In [10]:
merge_consecutive_annotations(raw)

✓ Merged 201 → 201 annotations (reduced by 0, 0.0%)


In [11]:
# Load bad channels from file and set them as bad
if old_bad_channels and os.path.exists(old_bad_channels):
    with open(old_bad_channels, 'r') as f:
        loaded_bad_channels = [line.strip() for line in f if line.strip()]
        loaded_bad_channels =[x for x in loaded_bad_channels if x in raw.ch_names]    
    print(f"success ({len(loaded_bad_channels)} channels)")
else:
    print(f"Bad channels file not found: {old_bad_channels}")

success (7 channels)


In [12]:
# Load bad channels from pipeline.txt if it exists
pipeline_file = f"F:/gennadiy/ella_processed/{sub}/pipeline.log"
if os.path.exists(pipeline_file):
    with open(pipeline_file, 'r') as f:
        content = f.read()
        
    if "Interpolated channels:" in content:
        for line in content.split('\n')[::-1]:
            if "Interpolated channels:" in line:
                list_str = line.split("Interpolated channels:")[-1].strip()
                try:
                    loaded_bad_channels = ast.literal_eval(list_str)
                    print(f"success ({len(loaded_bad_channels)} channels)")
                except Exception as e:
                    print(f"Error parsing channel list: {e}")
                break
    else:
        print("'Interpolated channels:' not found in pipeline file")
else:
    print(f"Pipeline file not found: {pipeline_file}")

Pipeline file not found: F:/gennadiy/ella_processed/SM09/pipeline.log


In [15]:
if loaded_bad_channels:
    print(f"original bad channels: {len(loaded_bad_channels)}")
    bads = [x for x in loaded_bad_channels if x in raw.ch_names]    
    # if not os.path.exists(pipeline_file):
    raw.info['bads'] = bads
    print(f"Loaded and set {len(bads)} bad channels from file:")
    # else:
    # print("X - not loading bad channels cause they are already interpolated")
    print(f"Bad channels: {sort_electrode_names(bads)}")
else:
    print("No bad channels loaded from file.")

original bad channels: 7
Loaded and set 7 bad channels from file:
Bad channels: ['E26', 'E56', 'E101', 'E159', 'E178', 'E191', 'E201']


### Look at the channels and mark bad ones!

In [18]:
raw.plot(duration=50)

In [19]:
bad_channels = sort_electrode_names([str(x) for x in raw.info['bads']])
bad_percentage = (len(bad_channels) / len(raw.ch_names)) * 100
print(f"Bad channels marked: {bad_channels}")
print(f"Count: {len(bad_channels)} ({bad_percentage:.2f}%)")

Bad channels marked: ['E8', 'E9', 'E44', 'E45', 'E56', 'E59', 'E89', 'E182', 'E184']
Count: 9 (5.11%)


In [20]:
with open(new_bad_channels, 'w') as f:
    for ch in bad_channels:
        f.write(f"{ch}\n")

In [21]:
_ = compare_raw_and_file_annotations(raw, f"{sub_dir}/annotations.txt")
# _ = compare_raw_and_file_annotations(raw, f"{sub_dir}/{sub}_annotations.txt")

Raw file annotations: 194
Text file annotations: 206
⚠️ Different number of annotations!


In [22]:
# Save current annotations to cleaned_annotations.txt
cleaned_annotations_path = f"{sub_dir}/{sub}_cleaned_annotations.txt"
raw.annotations.save(cleaned_annotations_path, overwrite=True)
print(f"Saved {len(raw.annotations)} annotations to: {cleaned_annotations_path}")

Saved 194 annotations to: I:/Shaked/ISO_data/MCI_clean/a_the_rest/SM09/SM09_cleaned_annotations.txt
